# Phase 1: Build and validate the analysis table

**CSE558 Data Science midterm project · Olist Brazilian e-commerce dataset**

**Research question.** Are late deliveries associated with lower customer review scores, and does the association persist when comparing orders with similar actual waiting times?

This notebook covers Phase 1 only:

1. Inspect the nine CSV files.
2. Define the unit of analysis.
3. Join the tables safely.
4. Engineer the features.
5. Build the encoded matrix that the 3,000,000 rule is checked on.

There are no hypothesis tests here.

**How to run.** Put the nine Olist CSVs in a folder called `data/` in the project root (the folder that contains `notebooks/`). Alternatively, put the Kaggle zip in the project root, and the first code cell will extract it. Then run *Kernel → Restart & Run All*. Every count is computed from the files; nothing is hard-coded.

**Source.** [Kaggle: Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce).

### Decisions taken in this notebook (each one is explained where it happens)

| # | Decision | Default chosen | Alternative kept for sensitivity |
|---|---|---|---|
| D1 | Which orders count as delivered | `order_status == 'delivered'` **and** a customer delivery timestamp exists | any status with a delivery timestamp |
| D2 | Orders with several review records | **exclude** them (keep orders with exactly one review) | keep the review with the latest `review_answer_timestamp` |
| D3 | Definition of "late" | delivered on a **later calendar day** than the estimated date | strict timestamp comparison |
| D4 | Reviews **answered** before the parcel arrived (`review_answer_timestamp` < `order_delivered_customer_date`) | **keep** them, with the flag `review_before_delivery` | drop them in a sensitivity test |
| D5 | One review record shared by several orders (a different issue from D2) | **keep** them, with the flag `review_shared_across_orders` | drop them in a sensitivity test |
| D6 | Orders with several products or sellers | sums and counts, plus **multi-hot** indicators (category, seller state); no single product is picked arbitrarily | — |

In [1]:
import os
from pathlib import Path
# All paths are relative to the project root. If Jupyter starts inside notebooks/, move up one level.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
import json
import zipfile

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DATA_DIR = Path("data")      # folder holding the nine CSVs
OUT_DIR = Path("outputs") / "phase1"    # everything this notebook writes goes here
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If the CSVs are not there yet but the Kaggle zip is, extract it.
if not (DATA_DIR / "olist_orders_dataset.csv").exists():
    zips = sorted(Path(".").glob("*.zip"))
    assert zips, "Put the Olist CSVs in ./data or the Kaggle zip in the project root (see README)."
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(DATA_DIR)

# Check that the CSVs are the exact version this project was built on (data_manifest.csv, if present)
if Path("data_manifest.csv").exists():
    import hashlib
    _m = pd.read_csv("data_manifest.csv")
    _missing = [f for f in _m.file if not (DATA_DIR / f).exists()]
    assert not _missing, f"missing source CSVs in {DATA_DIR}/: {_missing} (see README)"
    _bad = [r.file for r in _m.itertuples() if hashlib.sha256((DATA_DIR / r.file).read_bytes()).hexdigest() != r.sha256]
    assert not _bad, f"these CSVs differ from the version used for the project: {_bad}"
    print("All", len(_m), "source CSVs match data_manifest.csv")

# Decision switches (see the table above). Change these to run a sensitivity version.
REVIEW_RULE = "exactly_one"   # D2: "exactly_one" or "keep_latest"
LATE_RULE = "calendar_day"    # D3: "calendar_day" or "timestamp"

print("pandas", pd.__version__, "| numpy", np.__version__)

All 9 source CSVs match data_manifest.csv
pandas 3.0.1 | numpy 2.3.1


## 1A. Inspect the dataset

For every file we report the number of rows and columns, exact duplicate rows, and, for each column, its type, number of distinct values and missing values.

In [2]:
FILES = {
    "orders": "olist_orders_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
raw = {name: pd.read_csv(DATA_DIR / f) for name, f in FILES.items()}

overview = pd.DataFrame(
    [(n, FILES[n], d.shape[0], d.shape[1], int(d.duplicated().sum())) for n, d in raw.items()],
    columns=["table", "file", "rows", "columns", "exact_duplicate_rows"],
)
overview.to_csv(OUT_DIR / "01_original_dimensions.csv", index=False)
overview

,table,file,rows,columns,exact_duplicate_rows
0,orders,olist_orders_dataset.csv,99441,8,0
1,reviews,olist_order_reviews_dataset.csv,99224,7,0
2,items,olist_order_items_dataset.csv,112650,7,0
3,payments,olist_order_payments_dataset.csv,103886,5,0
4,customers,olist_customers_dataset.csv,99441,5,0
5,products,olist_products_dataset.csv,32951,9,0
6,sellers,olist_sellers_dataset.csv,3095,4,0
7,geolocation,olist_geolocation_dataset.csv,1000163,5,261831
8,translation,product_category_name_translation.csv,71,2,0


In [3]:
# Column-level profile of every file
profile = pd.concat(
    [
        pd.DataFrame({
            "table": n,
            "column": d.columns,
            "dtype": d.dtypes.astype(str).values,
            "n_unique": d.nunique().values,
            "n_missing": d.isna().sum().values,
        })
        for n, d in raw.items()
    ],
    ignore_index=True,
)
profile.to_csv(OUT_DIR / "01_column_profile.csv", index=False)
profile[profile.n_missing > 0]   # only the columns that have missing values

,table,column,dtype,n_unique,n_missing
4,orders,order_approved_at,str,90733,160
5,orders,order_delivered_carrier_date,str,81018,1783
6,orders,order_delivered_customer_date,str,95664,2965
11,reviews,review_comment_title,str,4527,87656
12,reviews,review_comment_message,str,36159,58247
33,products,product_category_name,str,73,610
34,products,product_name_lenght,float64,66,610
35,products,product_description_lenght,float64,2960,610
36,products,product_photos_qty,float64,19,610
37,products,product_weight_g,float64,2204,2


### Keys and relationships

A **primary key** identifies one row. A **foreign key** points to the primary key of another table. We test each candidate key in the data instead of assuming it.

In [4]:
def key_report(table, cols):
    d = raw[table]
    dup = int(d.duplicated(cols).sum())
    return {"table": table, "key": " + ".join(cols), "rows": len(d), "duplicate_keys": dup, "is_unique": dup == 0}

keys = pd.DataFrame([
    key_report("orders", ["order_id"]),
    key_report("orders", ["customer_id"]),
    key_report("customers", ["customer_id"]),
    key_report("customers", ["customer_unique_id"]),
    key_report("products", ["product_id"]),
    key_report("sellers", ["seller_id"]),
    key_report("translation", ["product_category_name"]),
    key_report("items", ["order_id", "order_item_id"]),
    key_report("payments", ["order_id", "payment_sequential"]),
    key_report("reviews", ["review_id"]),
    key_report("reviews", ["order_id"]),
    key_report("reviews", ["review_id", "order_id"]),
])
keys

,table,key,rows,duplicate_keys,is_unique
0,orders,order_id,99441,0,True
1,orders,customer_id,99441,0,True
2,customers,customer_id,99441,0,True
3,customers,customer_unique_id,99441,3345,False
4,products,product_id,32951,0,True
5,sellers,seller_id,3095,0,True
6,translation,product_category_name,71,0,True
7,items,order_id + order_item_id,112650,0,True
8,payments,order_id + payment_sequential,103886,0,True
9,reviews,review_id,99224,814,False


In [5]:
def coverage(child, ccol, parent, pcol):
    c, p = raw[child][ccol], raw[parent][pcol]
    return {
        "foreign_key": f"{child}.{ccol} -> {parent}.{pcol}",
        "child_values_missing_in_parent": int((~c.dropna().isin(p)).sum()),
        "parent_rows_without_child": int((~p.isin(c)).sum()),
    }

fk = pd.DataFrame([
    coverage("orders", "customer_id", "customers", "customer_id"),
    coverage("items", "order_id", "orders", "order_id"),
    coverage("payments", "order_id", "orders", "order_id"),
    coverage("reviews", "order_id", "orders", "order_id"),
    coverage("items", "product_id", "products", "product_id"),
    coverage("items", "seller_id", "sellers", "seller_id"),
    coverage("products", "product_category_name", "translation", "product_category_name"),
])
print("Categories with no English translation:",
      sorted(set(raw["products"].product_category_name.dropna()) - set(raw["translation"].product_category_name)))
fk

Categories with no English translation: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


,foreign_key,child_values_missing_in_parent,parent_rows_without_child
0,orders.customer_id -> customers.customer_id,0,0
1,items.order_id -> orders.order_id,0,775
2,payments.order_id -> orders.order_id,0,1
3,reviews.order_id -> orders.order_id,0,768
4,items.product_id -> products.product_id,0,0
5,items.seller_id -> sellers.seller_id,0,0
6,products.product_category_name -> translation....,13,0


**What the checks show** (the counts are in the outputs above):

- `orders` is the centre of the dataset. Each order has exactly one `customer_id`; `customer_unique_id` identifies the real person, who can place several orders.
- `items`, `payments` and `reviews` are **one-to-many** with `orders`: one order can have several items, several payment records, and sometimes several reviews.
- `review_id` is **not** unique. The same review can be attached to more than one order (D5).
- `products` → `translation`: two Portuguese category names have no English entry, and some products have no category at all.
- `geolocation` has a large number of exact duplicate rows and several points per zip code. It is **not joined**, because our question does not need coordinates. It is a candidate for later distance features.

```
customers 1 ── 1 orders 1 ── * items * ── 1 products * ── 1 translation
                     │                * ── 1 sellers
                     ├── 1 ── * payments
                     └── 1 ── * reviews
```

## 1B. Unit of analysis and filtering

**Unit:** one delivered order with exactly one review.

The customer's rating is given per order, and lateness is defined per order. The order is therefore the level at which "late" and "satisfaction" meet.

Before filtering we parse the timestamps and check that they are plausible.

In [6]:
orders = raw["orders"].copy()
TS_COLS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
           "order_delivered_customer_date", "order_estimated_delivery_date"]
for c in TS_COLS:
    before_missing = orders[c].isna().sum()
    orders[c] = pd.to_datetime(orders[c], format="%Y-%m-%d %H:%M:%S")  # strict: fails loudly on bad values
    assert orders[c].isna().sum() == before_missing, f"{c}: some values could not be parsed"

est = orders.order_estimated_delivery_date
print("Estimated dates that carry a time of day:", int((est != est.dt.normalize()).sum()),
      "-> the estimated date is a calendar date (midnight)")

plausibility = {
    "delivered before purchase": (orders.order_delivered_customer_date < orders.order_purchase_timestamp).sum(),
    "estimated before purchase": (orders.order_estimated_delivery_date < orders.order_purchase_timestamp).sum(),
    "approved before purchase": (orders.order_approved_at < orders.order_purchase_timestamp).sum(),
    "handed to carrier after customer delivery": (orders.order_delivered_carrier_date > orders.order_delivered_customer_date).sum(),
    "status not 'delivered' but has a delivery date": ((orders.order_status != "delivered") & orders.order_delivered_customer_date.notna()).sum(),
    "status 'delivered' but no delivery date": ((orders.order_status == "delivered") & orders.order_delivered_customer_date.isna()).sum(),
}
print("Purchase dates run from", orders.order_purchase_timestamp.min(), "to", orders.order_purchase_timestamp.max())
pd.Series(plausibility, name="orders").to_frame()

Estimated dates that carry a time of day: 0 -> the estimated date is a calendar date (midnight)
Purchase dates run from 2016-09-04 21:15:19 to 2018-10-17 17:30:18


,orders
delivered before purchase,0
estimated before purchase,0
approved before purchase,0
handed to carrier after customer delivery,23
status not 'delivered' but has a delivery date,6
status 'delivered' but no delivery date,8


The carrier-date anomaly involves a timestamp we do not use, so it is reported but not changed. Nothing that we do use is impossible: no delivery is dated before its purchase.

### Filtering funnel

In [7]:
reviews = raw["reviews"].copy()
reviews["review_creation_date"] = pd.to_datetime(reviews.review_creation_date, format="%Y-%m-%d %H:%M:%S")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews.review_answer_timestamp, format="%Y-%m-%d %H:%M:%S")
n_reviews_per_order = reviews.order_id.value_counts()

funnel = []
def step(label, df, reason):
    prev = funnel[-1]["rows_after"] if funnel else len(df)
    funnel.append({"step": label, "rows_after": len(df), "removed": prev - len(df), "reason": reason})
    return df

d = step("0. all orders", orders, "")
d = step("1. status == delivered", d[d.order_status == "delivered"], "not delivered (shipped, canceled, unavailable, ...)")
d = step("2. delivery timestamp present", d[d.order_delivered_customer_date.notna()], "lateness and waiting time cannot be computed")
d = step("3. estimated date present", d[d.order_estimated_delivery_date.notna()], "lateness cannot be computed")
d = step("4. at least one review", d[d.order_id.isin(n_reviews_per_order.index)], "no review, so no outcome")
if REVIEW_RULE == "exactly_one":
    d = step("5. exactly one review", d[d.order_id.isin(n_reviews_per_order[n_reviews_per_order == 1].index)],
             "several reviews: ambiguous outcome (D2)")
eligible_orders = d.copy()

funnel = pd.DataFrame(funnel)
funnel.to_csv(OUT_DIR / "02_filter_funnel.csv", index=False)
funnel

,step,rows_after,removed,reason
0,0. all orders,99441,0,
1,1. status == delivered,96478,2963,"not delivered (shipped, canceled, unavailable,..."
2,2. delivery timestamp present,96470,8,lateness and waiting time cannot be computed
3,3. estimated date present,96470,0,lateness cannot be computed
4,4. at least one review,95824,646,"no review, so no outcome"
5,5. exactly one review,95299,525,several reviews: ambiguous outcome (D2)


**Why orders with several reviews are excluded (D2).**

Among those orders, the review scores often disagree. Picking one review, say the latest, would be a choice that the data cannot justify. Excluding these orders costs about half a percent of the sample and keeps the outcome unambiguous. Keeping the latest review instead is the sensitivity alternative.

### Reconciling with the figures in the comparison doc (95,824 eligible, 6,381 late)

We recompute the counts under each combination of rules instead of forcing a match.

In [8]:
delivered = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
any_status = orders[orders.order_delivered_customer_date.notna()]

def late_flags(df):
    ts = df.order_delivered_customer_date > df.order_estimated_delivery_date
    day = df.order_delivered_customer_date.dt.normalize() > df.order_estimated_delivery_date
    return int(ts.sum()), int(day.sum())

rows = []
for base_name, base in [("status delivered + date (D1)", delivered), ("any status + date", any_status)]:
    for rule_name, ids in [("exactly one review (D2)", n_reviews_per_order[n_reviews_per_order == 1].index),
                           ("at least one review, keep one", n_reviews_per_order.index)]:
        sub = base[base.order_id.isin(ids)]
        ts, day = late_flags(sub)
        rows.append({"delivered rule": base_name, "review rule": rule_name, "orders": len(sub),
                     "late (timestamp)": ts, "late (calendar day, D3)": day})
reconcile = pd.DataFrame(rows)
reconcile.to_csv(OUT_DIR / "02_reconciliation.csv", index=False)
reconcile

,delivered rule,review rule,orders,late (timestamp),"late (calendar day, D3)"
0,status delivered + date (D1),exactly one review (D2),95299,7622,6353
1,status delivered + date (D1),"at least one review, keep one",95824,7661,6381
2,any status + date,exactly one review (D2),95305,7623,6354
3,any status + date,"at least one review, keep one",95830,7662,6382


**What the table shows.** A looser eligibility rule gives 95,824 eligible orders and 6,381 late orders. These counts come from:

- delivered orders that have **at least one** review, keeping one review per order; and
- lateness measured in **calendar days**.

That looser rule keeps one review even where an order has several. Under our stricter rule D2 (exactly one review per order), the eligible count is lower, and the late count is lower to match.

### Choosing the lateness rule (D3)

The estimated delivery date has no time of day; it is stored as midnight. A strict timestamp comparison would therefore count a parcel delivered at 3 pm **on** the promised day as late. That is not how a customer reads "arrives by 12 March", so we use the calendar-day rule.

In [9]:
e = eligible_orders
on_day = (e.order_delivered_customer_date.dt.normalize() == e.order_estimated_delivery_date).sum()
print(f"Deliveries on the promised day itself: {on_day:,} -> on time under D3, late under the timestamp rule")

Deliveries on the promised day itself: 1,269 -> on time under D3, late under the timestamp rule


### Sensitivity view of D2: exactly one review vs keep the latest answer

Under the alternative rule, an order with several reviews keeps the review whose `review_answer_timestamp` is latest. This is the customer's most recent opinion. The table puts the key counts side by side, using the corrected review-timing definition: the answer is compared with the delivery timestamp.

In [10]:
multi = n_reviews_per_order[n_reviews_per_order > 1].index
in_delivered = delivered.order_id.isin(multi)
print(f"Delivered orders with several review records (excluded under D2): {int(in_delivered.sum()):,}")

def rule_counts(review_table, name):
    j = delivered.merge(review_table[["order_id", "review_answer_timestamp", "review_score"]], on="order_id", how="inner",
                        validate="one_to_one")
    late_d = j.order_delivered_customer_date.dt.normalize() > j.order_estimated_delivery_date
    before = j.review_answer_timestamp < j.order_delivered_customer_date
    return {"review rule": name, "orders": len(j), "late": int(late_d.sum()), "late %": round(late_d.mean() * 100, 2),
            "answered before delivery": int(before.sum()), "late & answered before": int((late_d & before).sum()),
            "mean score, late": round(j.review_score[late_d].mean(), 3),
            "mean score, on time": round(j.review_score[~late_d].mean(), 3)}

exactly_one = reviews[~reviews.order_id.isin(multi)]
latest = reviews.sort_values(["order_id", "review_answer_timestamp"]).drop_duplicates("order_id", keep="last")
d2_sensitivity = pd.DataFrame([rule_counts(exactly_one, "exactly one (D2, primary)"),
                               rule_counts(latest, "keep latest answer (sensitivity)")])
d2_sensitivity.to_csv(OUT_DIR / "02_d2_sensitivity.csv", index=False)
d2_sensitivity

Delivered orders with several review records (excluded under D2): 525


,review rule,orders,late,late %,answered before delivery,late & answered before,"mean score, late","mean score, on time"
0,"exactly one (D2, primary)",95299,6353,6.67,4627,4450,2.273,4.291
1,keep latest answer (sensitivity),95824,6381,6.66,4653,4473,2.270,4.290


## 1C. Joining without duplicating orders

**The rule.** First aggregate every one-to-many table to **one row per order**, then join.

If we joined `items` directly, an order with three items would appear three times. Its review score would then be counted three times in every test. That inflates the sample size and gives multi-item orders extra weight.

Every join below uses pandas' `validate=` argument, which raises an error if the cardinality is not what we claim. Each join's rows before and after, unmatched rows and key uniqueness are written to a join log.

In [11]:
join_log = []
def logged_merge(left, right, on, how, validate, name):
    out = left.merge(right, on=on, how=how, validate=validate, indicator=True)
    join_log.append({
        "join": name, "keys": on, "how": how, "cardinality": validate,
        "rows_before": len(left), "rows_after": len(out),
        "left_unmatched": int((out._merge == "left_only").sum()),
        "order_id_unique_after": bool(out.order_id.is_unique) if "order_id" in out else None,
    })
    return out.drop(columns="_merge")

In [12]:
# Step 1: eligible orders + their single review (one-to-one)
rev_cols = ["order_id", "review_id", "review_score", "review_comment_title", "review_comment_message",
            "review_creation_date", "review_answer_timestamp"]
rev = reviews[rev_cols]
if REVIEW_RULE == "keep_latest":
    rev = rev.sort_values(["order_id", "review_answer_timestamp"]).drop_duplicates("order_id", keep="last")
rev = rev[rev.order_id.isin(eligible_orders.order_id)]

base = logged_merge(eligible_orders, rev, "order_id", "inner", "one_to_one", "orders x reviews")

# Flag reviews that are shared by more than one order in the full review table (D5)
shared_ids = reviews.review_id.value_counts()
base["review_shared_across_orders"] = base.review_id.isin(shared_ids[shared_ids > 1].index).astype(int)

# Step 2: + customer (one-to-one: one customer_id per order)
base = logged_merge(base, raw["customers"], "customer_id", "left", "one_to_one", "orders x customers")

### Item-level table, then aggregated to orders (D6)

First we attach product, category and seller details to each **item**. Each item maps to one product and one seller, so these joins are many-to-one. Then we collapse the items to one row per order:

- **Amounts** (`items_price_total`, `freight_total`, `weight_total_g`, `volume_total_cm3`) are **sums**. Each item row is one unit sold.
- **Counts** (`n_items`, `n_products`, `n_sellers`, `n_categories`) describe how complex the order is.
- **Category and seller state** become **multi-hot** indicators. "Order contains a `bed_bath_table` item" is 1 or 0, so an order with two categories gets two 1s instead of an arbitrary pick.
- For readable EDA labels only, `dominant_category` is the category with the largest share of the order's item price. Ties are broken alphabetically.

In [13]:
products = raw["products"].copy()
translation = raw["translation"].copy()
# Two categories have no English entry; we add a documented translation instead of losing them.
extra = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_food_preparers"],
})
translation = pd.concat([translation, extra], ignore_index=True)
products["volume_cm3"] = products.product_length_cm * products.product_height_cm * products.product_width_cm

items = raw["items"][raw["items"].order_id.isin(base.order_id)].copy()
n_item_rows = len(items)
items = items.merge(products, on="product_id", how="left", validate="many_to_one")
items = items.merge(translation, on="product_category_name", how="left", validate="many_to_one")
items = items.merge(raw["sellers"], on="seller_id", how="left", validate="many_to_one")
assert len(items) == n_item_rows, "an item-level join duplicated rows"
items["category"] = items.product_category_name_english.fillna("unknown")   # products with no category
print("item rows:", n_item_rows, "| items with unknown category:", int((items.category == "unknown").sum()),
      "| items with missing weight or size:", int(items[["product_weight_g", "volume_cm3"]].isna().any(axis=1).sum()))

g = items.groupby("order_id")
item_agg = pd.DataFrame({
    "n_items": g.size(),
    "n_products": g.product_id.nunique(),
    "n_sellers": g.seller_id.nunique(),
    "n_categories": g.category.nunique(),
    "n_seller_states": g.seller_state.nunique(),
    "items_price_total": g.price.sum(),
    "freight_total": g.freight_value.sum(),
    # .where(count == size): if any item has a missing weight or size, the order total stays missing
    "weight_total_g": g.product_weight_g.sum(min_count=1).where(g.product_weight_g.count() == g.size()),
    "volume_total_cm3": g.volume_cm3.sum(min_count=1).where(g.volume_cm3.count() == g.size()),
    "seller_states": g.seller_state.agg(lambda s: "|".join(sorted(set(s)))),
}).reset_index()

dom = (items.groupby(["order_id", "category"], as_index=False).price.sum()
            .sort_values(["order_id", "price", "category"], ascending=[True, False, True])
            .drop_duplicates("order_id")[["order_id", "category"]]
            .rename(columns={"category": "dominant_category"}))
item_agg = item_agg.merge(dom, on="order_id", how="left", validate="one_to_one")

cat_hot = pd.crosstab(items.order_id, items.category).clip(upper=1).add_prefix("cat__")
sstate_hot = pd.crosstab(items.order_id, items.seller_state).clip(upper=1).add_prefix("seller_state__")
print("category indicators:", cat_hot.shape[1], "| seller-state indicators:", sstate_hot.shape[1])

item rows: 108725 | items with unknown category: 1517 | items with missing weight or size: 18
category indicators: 74 | seller-state indicators: 22


### Payments, aggregated to orders

An order can be paid in several parts, for example a voucher plus a card. We sum the values, count the records, keep the maximum number of instalments, and record which payment types were used as multi-hot indicators.

In [14]:
pay = raw["payments"][raw["payments"].order_id.isin(base.order_id)]
gp = pay.groupby("order_id")
pay_agg = pd.DataFrame({
    "payment_total": gp.payment_value.sum(),
    "n_payment_records": gp.size(),
    "max_installments": gp.payment_installments.max(),
    "payment_types": gp.payment_type.agg(lambda s: "|".join(sorted(set(s)))),
}).reset_index()
pay_hot = pd.crosstab(pay.order_id, pay.payment_type).clip(upper=1).add_prefix("pay__")
print("payment-type indicators:", list(pay_hot.columns))

payment-type indicators: ['pay__boleto', 'pay__credit_card', 'pay__debit_card', 'pay__voucher']


In [15]:
# Step 3-4: attach the order-level aggregates (each is one row per order, so one-to-one)
table = logged_merge(base, item_agg, "order_id", "left", "one_to_one", "orders x item aggregates")
table = logged_merge(table, pay_agg, "order_id", "left", "one_to_one", "orders x payment aggregates")
table["payment_missing"] = table.payment_total.isna().astype(int)

join_log = pd.DataFrame(join_log)
join_log.to_csv(OUT_DIR / "03_join_log.csv", index=False)

# The core invariant of Phase 1
assert table.order_id.is_unique
assert len(table) == table.order_id.nunique() == len(eligible_orders)
print(f"Analysis table: {len(table):,} rows = {table.order_id.nunique():,} unique orders  ✓")
join_log

Analysis table: 95,299 rows = 95,299 unique orders  ✓


,join,keys,how,cardinality,rows_before,rows_after,left_unmatched,order_id_unique_after
0,orders x reviews,order_id,inner,one_to_one,95299,95299,0,True
1,orders x customers,customer_id,left,one_to_one,95299,95299,0,True
2,orders x item aggregates,order_id,left,one_to_one,95299,95299,0,True
3,orders x payment aggregates,order_id,left,one_to_one,95299,95299,1,True


## 1D. Feature engineering

All durations are in **days**: seconds ÷ 86,400, so 1.5 means one and a half days. Values are not clipped. Long or negative values are reported, and the flags show where they occur.

In [16]:
t = table
dc = t.order_delivered_customer_date
DAY = 86400

# Delivery features
t["delay_days"] = (dc - t.order_estimated_delivery_date).dt.total_seconds() / DAY          # negative = early
t["delay_calendar_days"] = (dc.dt.normalize() - t.order_estimated_delivery_date).dt.days    # whole days, matches D3
t["wait_days"] = (dc - t.order_purchase_timestamp).dt.total_seconds() / DAY
t["promised_days"] = (t.order_estimated_delivery_date - t.order_purchase_timestamp).dt.total_seconds() / DAY
if LATE_RULE == "calendar_day":
    t["is_late"] = (t.delay_calendar_days > 0).astype(int)
else:
    t["is_late"] = (t.delay_days > 0).astype(int)

# Review features (the outcomes)
msg = t.review_comment_message.fillna("").str.strip()
t["low_score"] = (t.review_score <= 2).astype(int)
t["has_comment"] = (msg != "").astype(int)
t["comment_len_chars"] = msg.str.len()                     # characters after trimming spaces; 0 = no comment
t["has_title"] = (t.review_comment_title.fillna("").str.strip() != "").astype(int)
# Review timing (D4). Three different timestamps are involved:
#   review_creation_date    = when Olist created / sent the satisfaction survey (almost always a date at midnight)
#   review_answer_timestamp = when the customer submitted the answer
#   order_delivered_customer_date = when the parcel actually arrived
# "Answered before delivery" compares the customer's answer with the arrival, using full timestamps.
t["review_lag_days"] = (t.review_answer_timestamp - dc).dt.total_seconds() / DAY   # negative = answered before arrival
t["review_before_delivery"] = (t.review_answer_timestamp < dc).astype(int)          # D4
# Survey timing is kept as a separate, clearly named feature. The survey date has no time of day,
# so it is compared with the delivery calendar date.
t["survey_lag_days"] = (t.review_creation_date.dt.normalize() - dc.dt.normalize()).dt.days  # negative = survey sent before arrival day
t["survey_sent_before_delivery"] = (t.survey_lag_days < 0).astype(int)
assert (t.review_answer_timestamp >= t.review_creation_date).all(), "an answer precedes its survey"

# Order and customer features
t["freight_ratio"] = t.freight_total / t.items_price_total    # share of freight relative to the goods
t["same_state"] = [int(cs == ss) for cs, ss in zip(t.customer_state, t.seller_states)]  # 1 = every seller is in the customer's state

# Time features (from the purchase timestamp)
pt = t.order_purchase_timestamp
t["purchase_year"] = pt.dt.year
t["purchase_month"] = pt.dt.month
t["purchase_yearmonth"] = pt.dt.strftime("%Y-%m")
t["purchase_weekday"] = pt.dt.day_name().str[:3]
t["purchase_hour"] = pt.dt.hour
t["is_weekend"] = pt.dt.dayofweek.ge(5).astype(int)

assert (t.is_late == (t.delay_calendar_days > 0)).all() or LATE_RULE != "calendar_day"
print(f"Late: {t.is_late.sum():,} ({t.is_late.mean():.2%}) | on time or early: {(1 - t.is_late).sum():,}")
print(f"delay_days > 0 but is_late = 0 (delivered later in the day on the promised date): "
      f"{((t.delay_days > 0) & (t.is_late == 0)).sum():,}")

Late: 6,353 (6.67%) | on time or early: 88,946
delay_days > 0 but is_late = 0 (delivered later in the day on the promised date): 1,269


**`delay_days` and `is_late` measure different things.** The estimated date is stored as midnight. A parcel that arrives at 3 pm on the promised day therefore has `delay_days` = +0.625, yet `is_late` = 0, because it arrived on the promised calendar day (D3). `delay_calendar_days` is the whole-day version that agrees exactly with `is_late`. Use `delay_days` as a continuous duration and `is_late` as the yes/no exposure; do not read a small positive `delay_days` as "late".

**Review timing uses the customer's answer, not the survey date.** `review_creation_date` is when Olist *sent* the survey; `review_answer_timestamp` is when the customer *answered* it. Only the answer tells us whether the customer had the parcel in hand when rating. So `review_before_delivery` compares the answer timestamp with the actual delivery timestamp, both with their times of day. The survey's own timing is kept separately as `survey_sent_before_delivery` and is never described as the time of the review.

### Checks on the derived variables

In [17]:
num_check = ["delay_days", "delay_calendar_days", "wait_days", "promised_days", "review_score", "comment_len_chars",
             "review_lag_days", "survey_lag_days", "items_price_total", "freight_total", "freight_ratio", "n_items",
             "n_sellers", "payment_total", "max_installments", "weight_total_g", "volume_total_cm3"]
checks = t[num_check].describe().T[["count", "mean", "min", "50%", "max"]]
checks["missing"] = t[num_check].isna().sum()
checks.round(2)

,count,mean,min,50%,max,missing
delay_days,95299.0,-11.21,-146.02,-11.97,188.98,0
delay_calendar_days,95299.0,-11.91,-147.00,-12.00,188.00,0
wait_days,95299.0,12.51,0.53,10.20,208.35,0
promised_days,95299.0,23.72,2.01,23.23,155.14,0
review_score,95299.0,4.16,1.00,5.00,5.00,0
comment_len_chars,95299.0,27.24,0.00,0.00,208.00,0
review_lag_days,95299.0,2.91,-186.76,1.92,518.98,0
survey_lag_days,95299.0,0.46,-186.00,1.00,67.00,0
items_price_total,95299.0,136.94,0.85,86.70,13440.00,0
freight_total,95299.0,22.77,0.00,17.17,1794.96,0


In [18]:
flags = {
    "wait_days <= 0": (t.wait_days <= 0).sum(),
    "wait_days > 60": (t.wait_days > 60).sum(),
    "promised_days <= 0": (t.promised_days <= 0).sum(),
    "delay over 30 days late": (t.delay_days > 30).sum(),
    "orders with a zero-weight product": items.loc[items.product_weight_g == 0, "order_id"].nunique(),
    "payment_total != price + freight (by > 1 BRL)": ((t.payment_total - t.items_price_total - t.freight_total).abs() > 1).sum(),
    "orders with no payment record": t.payment_missing.sum(),
    "comment present but only spaces (counted as no comment)": (t.review_comment_message.notna() & (t.has_comment == 0)).sum(),
    "delay_days > 0 but is_late = 0 (promised day)": ((t.delay_days > 0) & (t.is_late == 0)).sum(),
    "review answered before delivery (D4)": t.review_before_delivery.sum(),
    "review answered > 60 days after delivery": (t.review_lag_days > 60).sum(),
    "survey sent before the delivery day": t.survey_sent_before_delivery.sum(),
    "review record shared with another order (D5)": t.review_shared_across_orders.sum(),
}
flags_df = pd.Series(flags, name="orders").to_frame()
flags_df.to_csv(OUT_DIR / "03_quality_flags.csv")
flags_df

,orders
wait_days <= 0,0
wait_days > 60,290
promised_days <= 0,0
delay over 30 days late,341
orders with a zero-weight product,6
payment_total != price + freight (by > 1 BRL),245
orders with no payment record,1
comment present but only spaces (counted as no comment),26
delay_days > 0 but is_late = 0 (promised day),1269
review answered before delivery (D4),4627


In [19]:
# Original missingness in the analysis table, BEFORE any imputation for the encoded matrix
missing_table = t.isna().sum()
missing_table = missing_table[missing_table > 0].rename("missing_in_analysis_table").to_frame()
missing_table.to_csv(OUT_DIR / "03_analysis_table_missingness.csv")
missing_table

,missing_in_analysis_table
order_approved_at,14
order_delivered_carrier_date,1
review_comment_title,84129
review_comment_message,56575
weight_total_g,16
volume_total_cm3,16
payment_total,1
n_payment_records,1
max_installments,1
payment_types,1


In [20]:
# D4 matters for the research question: how do reviews answered before arrival split by lateness?
ct = pd.crosstab(t.is_late, t.review_before_delivery, margins=True)
ct.index = ["on time", "late", "total"]; ct.columns = ["answered after delivery", "answered before delivery", "total"]
grp = t.groupby(["is_late", "review_before_delivery"]).review_score.agg(["size", "mean", "std"])
grp["low_score_share"] = t.groupby(["is_late", "review_before_delivery"]).low_score.mean()
grp.index = grp.index.map(lambda k: f"{'late' if k[0] else 'on time'} / answered {'before' if k[1] else 'after'} delivery")
grp = grp.round(3)
grp.to_csv(OUT_DIR / "03_review_timing_groups.csv")
late = t[t.is_late == 1]
print(f"Answered before delivery: {t.review_before_delivery.sum():,} of {len(t):,} orders; "
      f"among late orders {late.review_before_delivery.sum():,} of {len(late):,} ({late.review_before_delivery.mean():.1%})")
display(ct); display(grp)

Answered before delivery: 4,627 of 95,299 orders; among late orders 4,450 of 6,353 (70.0%)


,answered after delivery,answered before delivery,total
on time,88769,177,88946
late,1903,4450,6353
total,90672,4627,95299


,size,mean,std,low_score_share
on time / answered after delivery,88769,4.292,1.148,0.092
on time / answered before delivery,177,3.938,1.443,0.175
late / answered after delivery,1903,3.723,1.394,0.194
late / answered before delivery,4450,1.653,1.190,0.807


In [21]:
# How the corrected definition compares with the earlier, survey-date-based one
old_def = (t.review_creation_date < dc.dt.normalize()).astype(int)   # earlier version: survey date vs delivery date
comparison = pd.crosstab(old_def.rename("old: survey date before delivery date"),
                         t.review_before_delivery.rename("new: answer before delivery timestamp"))
display(comparison)

# When is the survey sent? Days from the promised date and from the delivery date (survey date only)
survey_vs_promise = (t.review_creation_date.dt.normalize() - t.order_estimated_delivery_date).dt.days
timing = pd.DataFrame({
    "median survey date - promised date (days)": survey_vs_promise.groupby(t.is_late).median(),
    "median survey date - delivery date (days)": t.survey_lag_days.groupby(t.is_late).median(),
    "median answer - delivery (days)": t.review_lag_days.groupby(t.is_late).median().round(2),
})
timing.index = ["on time", "late"]
timing

new: answer before delivery timestamp,0,1
old: survey date before delivery date,,
0,90159,200
1,513,4427


,median survey date - promised date (days),median survey date - delivery date (days),median answer - delivery (days)
on time,-12.0,1.0,2.02
late,2.0,-4.0,-2.75


**What the corrected figures show.** Most late orders are answered **before the parcel arrives**, and those answers are much lower than any other group. Two things changed when we moved from the survey date to the answer timestamp:

1. Some customers received the survey before the parcel arrived but answered **after** it arrived. They used to be counted as "before" and now count as "after".
2. Some surveys were created on the delivery day, which the old date-level comparison counted as "not before". The customer answered earlier that same day, before the parcel arrived, so these now count as "before".

The timing table explains the mechanism without assuming it. For on-time orders, the survey is typically created about a day after delivery. For late orders, it is created a couple of days after the promised date, which is several days before the parcel arrives.

This is part of what "late" means for a customer, so these orders are kept (D4). The flag lets Phase 4 test whether the late penalty survives among reviews answered after arrival.

Payment totals that differ from price plus freight are rare and are probably due to vouchers or instalment interest. We report them and leave them unchanged.

## 1E. Encoding and the 3,000,000 requirement

The **feature matrix `X`** holds order characteristics that are known when the order is placed or delivered. The following are kept **out** of `X`:

- **Identifiers** (`order_id`, `customer_id`, ...). They are labels, not information about the order.
- **Outcomes** (`review_score`, `low_score`, `has_comment`, `comment_len_chars`, `has_title`) and review-timing variables. They go in a separate outcomes file, so no later model can learn from the answer.
- **Raw text and raw timestamps.** They are kept in the analysis table instead.
- **`customer_city`** (thousands of levels; too sparse) and **geolocation**.

Encoding:

- **One-hot** for single-valued categoricals: customer state, purchase month, purchase weekday. Every level is kept. For a regression later, one level per group is dropped as the reference.
- **Multi-hot** for multi-valued ones: product category, seller state, payment type.
- **Numeric columns** are used as they are. The few missing values are filled with the median, and each fill is reported.
- Columns that are **constant** in this sample carry no information. They are dropped *before* counting.

**Important: `X` documents the dimension requirement; it is not a ready-made modelling feature set.** It deliberately contains the delivery variables we study (`is_late`, `delay_days`, `wait_days`). For any later predictive model, the target and leakage-prone columns must be chosen per question and removed. For example, a model predicting `is_late` must drop `is_late`, `delay_days` and `wait_days`, because they are computed from the actual delivery date. The statistical tests in Phase 4 use a small, explicitly named set of variables, not this matrix.

In [22]:
NUMERIC = ["is_late", "delay_days", "wait_days", "promised_days", "items_price_total", "freight_total",
           "freight_ratio", "n_items", "n_products", "n_sellers", "n_categories", "n_seller_states",
           "payment_total", "n_payment_records", "max_installments", "weight_total_g", "volume_total_cm3",
           "same_state", "purchase_year", "purchase_hour", "is_weekend"]
OUTCOMES = ["review_score", "low_score", "has_comment", "comment_len_chars", "has_title",
            "review_before_delivery", "review_lag_days", "survey_sent_before_delivery", "survey_lag_days",
            "review_shared_across_orders"]

X_num = t.set_index("order_id")[NUMERIC].copy()
imputed = {}
for c in NUMERIC:
    n = int(X_num[c].isna().sum())
    if n:
        imputed[c] = n
        X_num[c] = X_num[c].fillna(X_num[c].median())
print("Median-imputed values:", imputed)

one_hot = pd.get_dummies(
    t.set_index("order_id")[["customer_state", "purchase_month", "purchase_weekday"]].astype(str),
    prefix=["cust_state", "month", "wday"], dtype=int)

idx = X_num.index
blocks = {
    "numeric": X_num,
    "customer state (one-hot)": one_hot.filter(like="cust_state_"),
    "purchase month (one-hot)": one_hot.filter(like="month_"),
    "purchase weekday (one-hot)": one_hot.filter(like="wday_"),
    "product category (multi-hot)": cat_hot.reindex(idx, fill_value=0),
    "seller state (multi-hot)": sstate_hot.reindex(idx, fill_value=0),
    "payment type (multi-hot)": pay_hot.reindex(idx, fill_value=0),
}
X = pd.concat(blocks.values(), axis=1)

constant = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=constant)
print("Dropped constant columns:", constant)

assert X.index.is_unique and len(X) == len(t)
assert not X.isna().any().any(), "X still has missing values"

block_sizes = pd.Series({k: len([c for c in v.columns if c in X.columns]) for k, v in blocks.items()}, name="columns")
display(block_sizes.to_frame())

Median-imputed values: {'payment_total': 1, 'n_payment_records': 1, 'max_installments': 1, 'weight_total_g': 16, 'volume_total_cm3': 16}
Dropped constant columns: []


,columns
numeric,21
customer state (one-hot),27
purchase month (one-hot),12
purchase weekday (one-hot),7
product category (multi-hot),74
seller state (multi-hot),22
payment type (multi-hot),4


In [23]:
n_inst, n_attr = X.shape
product = n_inst * n_attr
print("Encoded feature matrix X")
print(f"  instances  = {n_inst:,}")
print(f"  attributes = {n_attr:,}")
print(f"  {n_inst:,} x {n_attr:,} = {product:,}")
print(f"  >= 3,000,000 ?  {'YES' if product >= 3_000_000 else 'NO'}  ({product / 3_000_000:.2f}x the requirement)")
print(f"  minimum attributes needed at this many instances: {int(np.ceil(3_000_000 / n_inst))}")

n_out = len(OUTCOMES)
print(f"\nWith the {n_out} outcome columns added (full encoded table): "
      f"{n_inst:,} x {n_attr + n_out:,} = {n_inst * (n_attr + n_out):,}")
assert product >= 3_000_000

Encoded feature matrix X
  instances  = 95,299
  attributes = 167
  95,299 x 167 = 15,914,933
  >= 3,000,000 ?  YES  (5.30x the requirement)
  minimum attributes needed at this many instances: 32

With the 10 outcome columns added (full encoded table): 95,299 x 177 = 16,867,923


## 1F. Save the outputs

In [24]:
outcomes = t.set_index("order_id")[OUTCOMES]

# The analysis table: one row per order, raw columns kept so every feature can be audited
table_cols = (["order_id", "customer_id", "customer_unique_id", "review_id", "order_status"] + TS_COLS +
              ["review_creation_date", "review_answer_timestamp", "customer_city", "customer_state",
               "seller_states", "dominant_category", "payment_types", "purchase_yearmonth", "purchase_month",
               "purchase_weekday", "delay_calendar_days", "payment_missing"] +
              [c for c in NUMERIC if c not in ("purchase_year",)] + ["purchase_year"] + OUTCOMES +
              ["review_comment_title", "review_comment_message"])
table_cols = list(dict.fromkeys(table_cols))   # remove repeats, keep order
analysis = t[table_cols]

analysis.to_csv(OUT_DIR / "analysis_table.csv", index=False)
X.to_csv(OUT_DIR / "X_encoded.csv")
outcomes.to_csv(OUT_DIR / "outcomes.csv")
try:
    analysis.to_parquet(OUT_DIR / "analysis_table.parquet", index=False)
    X.to_parquet(OUT_DIR / "X_encoded.parquet")
    print("Saved CSV and Parquet versions.")
except ImportError:
    print("pyarrow not installed: saved CSV only.")

# Check that what was written can be read back unchanged in shape
back = pd.read_csv(OUT_DIR / "analysis_table.csv")
assert back.shape == analysis.shape and back.order_id.is_unique
print("analysis_table:", analysis.shape, "| X_encoded:", X.shape, "| outcomes:", outcomes.shape)

Saved CSV and Parquet versions.
analysis_table: (95299, 55) | X_encoded: (95299, 167) | outcomes: (95299, 10)


In [25]:
FEATURES = [
    ("is_late", "delivery", "1 if delivered on a later calendar day than the estimated date (D3)"),
    ("delay_days", "delivery", "delivered timestamp minus estimated date, in days; negative = early"),
    ("delay_calendar_days", "delivery", "delivery calendar date minus estimated date, whole days; >0 means late"),
    ("wait_days", "delivery", "delivered timestamp minus purchase timestamp, in days"),
    ("promised_days", "delivery", "estimated date minus purchase timestamp, in days (how long Olist promised)"),
    ("review_score", "outcome", "customer's 1-5 star rating (raw)"),
    ("low_score", "outcome", "1 if review_score is 1 or 2"),
    ("has_comment", "outcome/text", "1 if the comment text is non-empty after trimming spaces"),
    ("comment_len_chars", "outcome/text", "number of characters in the trimmed comment; 0 if none"),
    ("has_title", "outcome/text", "1 if the comment title is non-empty"),
    ("review_lag_days", "review timing", "review_answer_timestamp minus order_delivered_customer_date, in days (full timestamps); negative = answered before arrival"),
    ("review_before_delivery", "review timing", "1 if review_answer_timestamp is earlier than order_delivered_customer_date (D4)"),
    ("survey_lag_days", "survey timing", "review_creation_date minus delivery calendar date, whole days; when the survey was SENT, not answered"),
    ("survey_sent_before_delivery", "survey timing", "1 if the survey was created before the delivery calendar day"),
    ("review_shared_across_orders", "data quality", "1 if the same review_id is attached to another order (D5); different from an order having several reviews (D2)"),
    ("items_price_total", "order", "sum of item prices (BRL)"),
    ("freight_total", "order", "sum of freight charges (BRL)"),
    ("freight_ratio", "order", "freight_total / items_price_total"),
    ("n_items", "order", "number of item rows (units) in the order"),
    ("n_products", "order", "number of distinct products"),
    ("n_sellers", "order", "number of distinct sellers"),
    ("n_categories", "order", "number of distinct product categories"),
    ("n_seller_states", "order", "number of distinct seller states"),
    ("payment_total", "payment", "sum of payment values (BRL); missing for orders with no payment record"),
    ("n_payment_records", "payment", "number of payment records (split payments)"),
    ("max_installments", "payment", "largest number of instalments among the order's payments"),
    ("pay__<type>", "payment", "multi-hot: 1 if the order used this payment type"),
    ("weight_total_g", "product", "sum of product weights over items (g); missing if any item lacks weight"),
    ("volume_total_cm3", "product", "sum of length x height x width over items (cm3)"),
    ("cat__<category>", "product", "multi-hot: 1 if the order contains an item of this English category"),
    ("dominant_category", "product", "category with the largest share of item price (EDA label only)"),
    ("customer_state", "location", "customer's state (27 levels), one-hot in X"),
    ("seller_state__<UF>", "location", "multi-hot: 1 if any seller of the order is in this state"),
    ("same_state", "location", "1 if every seller is in the customer's state"),
    ("purchase_year / _month / _weekday / _hour", "time", "from the purchase timestamp; month and weekday one-hot in X"),
    ("purchase_yearmonth", "time", "YYYY-MM, for monthly trends"),
    ("is_weekend", "time", "1 if purchased on Saturday or Sunday"),
]
feature_dict = pd.DataFrame(FEATURES, columns=["feature", "group", "meaning and construction"])
feature_dict.to_csv(OUT_DIR / "04_feature_dictionary.csv", index=False)
feature_dict

,feature,group,meaning and construction
0,is_late,delivery,1 if delivered on a later calendar day than th...
1,delay_days,delivery,"delivered timestamp minus estimated date, in d..."
2,delay_calendar_days,delivery,"delivery calendar date minus estimated date, w..."
3,wait_days,delivery,"delivered timestamp minus purchase timestamp, ..."
4,promised_days,delivery,"estimated date minus purchase timestamp, in da..."
5,review_score,outcome,customer's 1-5 star rating (raw)
6,low_score,outcome,1 if review_score is 1 or 2
7,has_comment,outcome/text,1 if the comment text is non-empty after trimm...
8,comment_len_chars,outcome/text,number of characters in the trimmed comment; 0...
9,has_title,outcome/text,1 if the comment title is non-empty


## 1G. Summary

In [26]:
summary = {
    "original_orders": int(len(orders)),
    "eligible_orders": int(len(t)),
    "unique_order_ids": int(t.order_id.nunique()),
    "row_uniqueness_verified": bool(t.order_id.is_unique),
    "late_orders": int(t.is_late.sum()),
    "on_time_or_early_orders": int((t.is_late == 0).sum()),
    "late_share": round(float(t.is_late.mean()), 4),
    "answered_before_delivery_all": int(t.review_before_delivery.sum()),
    "late_answered_before_delivery": int(((t.is_late == 1) & (t.review_before_delivery == 1)).sum()),
    "late_share_answered_before_delivery": round(float(t.loc[t.is_late == 1, "review_before_delivery"].mean()), 4),
    "mean_score_by_group": {k: round(float(v), 3) for k, v in grp["mean"].items()},
    "X_instances": int(n_inst), "X_attributes": int(n_attr), "X_product": int(product),
    "review_rule": REVIEW_RULE, "late_rule": LATE_RULE,
    "imputed_values": imputed, "dropped_constant_columns": constant,
}
json.dump(summary, open(OUT_DIR / "05_phase1_summary.json", "w"), indent=2)
for k, v in summary.items():
    print(f"{k:32s} {v:,}" if isinstance(v, int) and not isinstance(v, bool) else f"{k:32s} {v}")

original_orders                  99,441
eligible_orders                  95,299
unique_order_ids                 95,299
row_uniqueness_verified          True
late_orders                      6,353
on_time_or_early_orders          88,946
late_share                       0.0667
answered_before_delivery_all     4,627
late_answered_before_delivery    4,450
late_share_answered_before_delivery 0.7005
mean_score_by_group              {'on time / answered after delivery': 4.292, 'on time / answered before delivery': 3.938, 'late / answered after delivery': 3.723, 'late / answered before delivery': 1.653}
X_instances                      95,299
X_attributes                     167
X_product                        15,914,933
review_rule                      exactly_one
late_rule                        calendar_day
imputed_values                   {'payment_total': 1, 'n_payment_records': 1, 'max_installments': 1, 'weight_total_g': 16, 'volume_total_cm3': 16}
dropped_constant_columns         []


### Methodological Notes & Key Design Rationales

**Why is the unit of analysis one order?**

The rating is given per order, and the promised and actual delivery dates are per order. Using items or payments as rows would count the same rating several times and make the observations look more numerous and more independent than they are.

**Why aggregate one-to-many tables before joining?**

A direct join repeats the order once per item or payment. A three-item order would then count three times in every test. The `validate=` checks and the final assertion `rows == unique order_id` prove this did not happen.

**Why exclude delivered orders with no delivery date?**

Without the actual delivery date we cannot tell whether the order was late, or how long the customer waited. Guessing a date would create the very variable we are testing.

**Why do multi-review orders need a rule?**

When an order has two reviews with different scores, its outcome is ambiguous. We exclude these orders (about 0.5% of the sample). Keeping the latest review is run as the alternative, so we can show the conclusion does not depend on this choice.

**Why does the 3M rule matter?**

It ensures the dataset is large and rich enough for the analysis. We show the arithmetic on the encoded matrix after removing identifiers, outcomes and constant columns, so the count is honest rather than padded.

**How do you know a review was given before the parcel arrived?**

Three timestamps are involved: when Olist *sent* the survey, when the customer *answered* it, and when the parcel *arrived*. Only the answer time tells us what the customer had in hand when they rated. So `review_before_delivery` = 1 when the answer timestamp is earlier than the delivery timestamp, compared to the minute. The survey date is a separate feature and is never called the review time. Using the answer timestamp rather than the survey creation date ensures accurate classification of pre-arrival reviews; both are compared side by side in this notebook.

**What is the difference between D2 and D5?**

D2 is one order with **several review records**: the outcome is ambiguous, so the order is excluded. D5 is one review record **attached to several orders**, probably a customer rating several orders from one purchase in a single answer. The outcome is clear, so these orders are kept. However, the same rating then appears for more than one order, which weakens the independence assumption. Excluding them is a planned sensitivity test in Phase 4.

**Additional empirical observations from data integration:**

- Most late orders are **answered before** the parcel arrives, and those reviews are far lower. This is a flag for Phase 4.
- Retaining the latest review for multi-review orders yields 95,824 orders (6,381 late), whereas our primary single-review filter yields 95,299 orders.
- A delivery on the promised day counts as on time. The estimated date has no time of day, which is why we chose the calendar-day rule.